# Spark Optimization & Performance Tuning
Sesi 16, Fullstack Bangalore

Notebook ini memakai transaksi dan pelanggan sintetis. Kita membaca execution plan,
membandingkan strategi join, mengatur partisi, mencoba AQE dan cache, memeriksa skew, mengamati spill, dan memadatkan file Parquet kecil.
Setiap percobaan membandingkan hasil dengan perhitungan referensi.

## Sebelum mulai
Gunakan Python 3.10–3.13, Java 17, dan PySpark 4.0.1. Buka notebook di VS Code
atau Jupyter, pilih kernel khusus lab, lalu jalankan sel dari atas ke bawah.
Petunjuk instalasi ada di `PETUNJUK_NOTEBOOK.md` pada folder yang sama.

Default latihan adalah **10.000 baris dan 3 pengukuran** per varian setelah warm-up.
Mulai dari ukuran ini. Setelah berhasil, ubah `N_ROWS` menjadi 300.000 dan gunakan
**Restart Kernel, lalu Run All** untuk mengulang seluruh percobaan.

Notebook menjalankan Spark lokal dengan dua thread. Hasil disimpan dalam folder baru
di `notebook_outputs`. Tidak ada koneksi ke MinIO, database, atau dataset eksternal.
Pengaturan environment hanya berlaku pada proses kernel dan dipulihkan di sel terakhir.


## 1. Istilah dan kapan digunakan

| Konsep | Pengertian | Kapan dipakai atau diperiksa |
|---|---|---|
| Transformation | Operasi pembentuk rencana, seperti filter dan select | Ketika menyusun pengolahan data |
| Action | Operasi yang meminta hasil, seperti collect atau count | Ketika menjalankan query dan mengukur waktu |
| Shuffle | Pertukaran data antarpartisi | Pada join, agregasi, dan repartition yang memerlukan distribusi ulang |
| Broadcast join | Penyalinan sisi kecil untuk join lokal pada task | Ketika sisi kecil muat di memori executor dan jenis join mendukung |
| Repartition | Pembagian ulang data, biasanya dengan shuffle | Saat perlu mengubah jumlah atau distribusi partisi |
| Coalesce | Pengurangan jumlah partisi tanpa shuffle penuh | Saat data mengecil dan jumlah partisi ingin dikurangi |
| AQE | Penyesuaian plan memakai statistik runtime | Untuk menyesuaikan partisi atau strategi yang didukung selama eksekusi |
| Cache/persist | Penyimpanan hasil perantara | Saat hasil yang mahal dihitung akan dipakai berulang |
| Data skew | Distribusi data atau beban yang tidak merata | Ketika sebagian task memproses data jauh lebih banyak |

Broadcast tetap membutuhkan pengiriman data. Cache membutuhkan biaya materialisasi.
Coalesce yang terlalu kecil membatasi paralelisme. Perubahan ini perlu diukur pada query yang sama.


## 2. Pemeriksaan Python dan Java
Sel berikut memeriksa paket serta Java yang terlihat pada PATH. Notebook memakai JAR
dari instalasi PySpark yang sama dengan kernel. Ini menghindari benturan dengan
`SPARK_HOME` dari proyek lain.

Jika pemeriksaan berhenti, ikuti pesan error, perbaiki environment, kemudian restart kernel.
Notebook menolak memakai SparkContext lain yang sudah aktif dalam kernel.


In [1]:
import os
import sys
import re
import io
import json
import csv
import shutil
import subprocess
import statistics
import contextlib
import platform
from pathlib import Path
from datetime import datetime
from time import perf_counter
from uuid import uuid4
from pprint import pprint

import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError('Gunakan kernel Python 3.10 sampai 3.13 untuk paket lab ini.')
if pyspark.__version__ != '4.0.1':
    raise RuntimeError(f'PySpark terbaca {pyspark.__version__}. Pilih kernel dengan PySpark 4.0.1.')
if SparkContext._active_spark_context is not None:
    raise RuntimeError('SparkContext sudah aktif. Restart kernel sebelum menjalankan notebook dari awal.')

java = shutil.which('java')
if java is None:
    raise RuntimeError('Java tidak ditemukan. Siapkan Java 17 pada PATH lalu restart kernel.')
probe = subprocess.run([java, '-XshowSettings:properties', '-version'],
                       capture_output=True, text=True, timeout=20)
info = probe.stdout + probe.stderr
java_home = re.search(r'^\s*java.home\s*=\s*(.+)$', info, re.MULTILINE)
java_major = re.search(r'^\s*java.specification.version\s*=\s*(\d+)', info, re.MULTILINE)
if probe.returncode != 0 or not java_home or not java_major or int(java_major.group(1)) < 17:
    raise RuntimeError('Java 17+ diperlukan. Periksa java -version pada terminal environment lab.')

env_updates = {'SPARK_HOME': str(Path(pyspark.__file__).resolve().parent),
               'JAVA_HOME': java_home.group(1).strip(),
               'SPARK_LOCAL_IP': '127.0.0.1', 'PYSPARK_PYTHON': sys.executable}
env_before = {key: os.environ.get(key) for key in env_updates}
os.environ.update(env_updates)
print('Python:', platform.python_version())
print('PySpark:', pyspark.__version__)
print('Java major:', java_major.group(1))
print('Pemeriksaan environment selesai.')


Python: 3.12.14
PySpark: 4.0.1
Java major: 17
Pemeriksaan environment selesai.


## 3. Ukuran latihan dan folder hasil
Setiap Run All membuat folder baru. Folder hasil lama tidak ditimpa atau dihapus.
Jalankan kernel dengan working directory di `sesi16`, `sesi16/lab`, atau folder induk materi.


In [2]:
N_ROWS = 10_000
REPEATS = 3
assert 1_000 <= N_ROWS <= 1_000_000, 'Untuk kelas, gunakan 1.000 sampai 1.000.000 baris.'
assert 1 <= REPEATS <= 10

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / 'lab', cwd / 'sesi16' / 'lab']
LAB_DIR = next((p for p in candidates if (p / 'spark_lab.py').is_file()), None)
if LAB_DIR is None:
    raise RuntimeError('Working directory belum sesuai. Buka notebook dari folder sesi16/lab.')
RUN_DIR = LAB_DIR / 'notebook_outputs' / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
completed_checks = []
print('Baris:', N_ROWS, '| Pengukuran per varian:', REPEATS)
print('Folder hasil:', RUN_DIR)

def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')


Baris: 10000 | Pengukuran per varian: 3
Folder hasil: F:\programming\ngajar\full_stack_bangalore\sesi 17 Data Lake MinIO\sesi16\lab\notebook_outputs\20261025_164931_d6c23457


## 4. Menjalankan Spark lokal
`local[2]` memakai dua thread pada komputer ini. Pengaturan join otomatis dan AQE
dimatikan untuk baseline agar perubahan antarpercobaan mudah diamati.
Ini pengaturan eksperimen, bukan rekomendasi konfigurasi produksi.

Spark UI dibatasi ke localhost. Untuk melihat UI secara langsung, jalankan notebook
per sel dan buka alamat yang dicetak sebelum menjalankan sel penutup.


In [ ]:
spark = (SparkSession.builder.appName('Sesi16-Notebook')
         .master('local[2]')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.driver.bindAddress', '127.0.0.1')
         .config('spark.ui.bindAddress', '127.0.0.1')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.adaptive.enabled', 'false')
         .config('spark.sql.autoBroadcastJoinThreshold', '-1')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

def configure(partitions=8, aqe=False):
    spark.conf.set('spark.sql.shuffle.partitions', str(partitions))
    spark.conf.set('spark.sql.adaptive.enabled', str(aqe).lower())
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
    spark.conf.set('spark.sql.adaptive.autoBroadcastJoinThreshold', '-1')

environment = {'python': platform.python_version(), 'spark': spark.version,
               'java': spark.sparkContext._jvm.java.lang.System.getProperty('java.version'),
               'master': spark.sparkContext.master, 'rows': N_ROWS, 'repeats': REPEATS,
               'ui': spark.sparkContext.uiWebUrl,
               'limitations': ['Data sintetis pada satu komputer',
                              'Cache OS dan kondisi JVM tidak direset antar-trial',
                              'Waktu lokal tidak mewakili cluster produksi']}
save_json('environment.json', environment)
print('Spark UI:', environment['ui'])
print('Master:', environment['master'])


Spark UI: http://127.0.0.1:4040
Master: local[2]


## 5. Data transaksi dan pelanggan
`orders` berisi nomor order, pelanggan, dan nilai transaksi. `customers` memetakan
1.000 pelanggan ke lima wilayah. Kita menghitung total penjualan dan jumlah order per wilayah.

Nilai transaksi berupa integer agar pemeriksaan hasil dapat dilakukan persis.
Referensi dihitung dengan Python biasa dari rumus data, tanpa memakai join Spark.


In [4]:
orders = (spark.range(N_ROWS, numPartitions=4)
          .select(F.col('id').alias('order_id'),
                  (F.col('id') % 1000).alias('customer_id'),
                  ((F.col('id') % 97 + 1) * 100).alias('amount')))
customers = (spark.range(1000, numPartitions=2)
             .select(F.col('id').alias('customer_id'),
                     (F.col('id') % 5).cast('int').alias('region')))
direct = orders.withColumn('region', (F.col('customer_id') % 5).cast('int'))

def aggregate(df):
    return (df.groupBy('region')
            .agg(F.sum('amount').alias('revenue'), F.count('*').alias('orders'))
            .select('region', 'revenue', 'orders'))

def canonical(rows):
    return sorted(tuple(row) for row in rows)

totals = {region: [0, 0] for region in range(5)}
cache_totals = {(k, region): [0, 0] for k in range(3) for region in range(5)}
for i in range(N_ROWS):
    region = (i % 1000) % 5
    amount = (i % 97 + 1) * 100
    totals[region][0] += amount
    totals[region][1] += 1
    cache_totals[(i % 3, region)][0] += amount
    cache_totals[(i % 3, region)][1] += 64
reference = [(region, *totals[region]) for region in range(5)]
cache_reference = [[(region, *cache_totals[(k, region)]) for region in range(5)] for k in range(3)]
assert canonical(aggregate(direct).collect()) == reference
save_json('reference.json', reference)
orders.show(5, truncate=False)
customers.show(5, truncate=False)
print('Referensi: (region, revenue, orders)')
pprint(reference)

completed_checks.append('reference')


+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|0       |0          |100   |
|1       |1          |200   |
|2       |2          |300   |
|3       |3          |400   |
|4       |4          |500   |
+--------+-----------+------+
only showing top 5 rows
+-----------+------+
|customer_id|region|
+-----------+------+
|0          |0     |
|1          |1     |
|2          |2     |
|3          |3     |
|4          |4     |
+-----------+------+
only showing top 5 rows
Referensi: (region, revenue, orders)
[(0, 9795600, 2000),
 (1, 9791900, 2000),
 (2, 9788200, 2000),
 (3, 9794200, 2000),
 (4, 9790500, 2000)]


## 6. Lazy evaluation dan execution plan
Pembentukan DataFrame menyusun rencana. `collect()` menjalankan action dan mengirim hasil
ke driver. Di sini hasilnya hanya lima baris, sehingga pengumpulan tersebut tetap kecil.

Pada plan, cari `Exchange` untuk perpindahan data dan `SortMergeJoin` untuk strategi join
baseline. `explain("formatted")` menampilkan operator fisik beserta rinciannya.


In [5]:
configure()
baseline_query = aggregate(orders.join(customers, 'customer_id', 'inner'))
baseline_query.explain('formatted')
started = perf_counter()
baseline_rows = baseline_query.collect()
first_seconds = perf_counter() - started
assert canonical(baseline_rows) == reference
print(f'Eksekusi pertama: {first_seconds:.4f} detik. Hasil sesuai referensi.')


== Physical Plan ==
* HashAggregate (14)
+- Exchange (13)
   +- * HashAggregate (12)
      +- * Project (11)
         +- * SortMergeJoin Inner (10)
            :- * Sort (5)
            :  +- Exchange (4)
            :     +- * Project (3)
            :        +- * Filter (2)
            :           +- * Range (1)
            +- * Sort (9)
               +- Exchange (8)
                  +- * Project (7)
                     +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 10000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 1000))

(3) Project [codegen id : 1]
Output [2]: [(id#0L % 1000) AS customer_id#2L, (((id#0L % 97) + 1) * 100) AS amount#3L]
Input [1]: [id#0L]

(4) Exchange
Input [2]: [customer_id#2L, amount#3L]
Arguments: hashpartitioning(customer_id#2L, 8), ENSURE_REQUIREMENTS, [plan_id=99]

(5) Sort [codegen id : 2]
Input [2]: [customer_id#2L, amount#3L]
Arguments: [customer_id#2L ASC 

Eksekusi pertama: 0.9956 detik. Hasil sesuai referensi.


## 7. Pengukuran yang sama untuk setiap varian
Fungsi berikut menjalankan warm-up, lalu mengukur setiap varian beberapa kali.
Urutan varian digeser pada setiap putaran. Timer mencakup action `collect()`.
Pembangunan query, pencetakan plan, dan pemeriksaan hasil berada di luar timer.

Median adalah waktu tengah setelah hasil diurutkan. Simpan juga minimum dan maksimum
agar variasi antar-run terlihat. Perbedaan waktu yang kecil belum tentu berarti perbaikan.


In [6]:
def capture_plan(query):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        query.explain('formatted')
    return buffer.getvalue()

all_trials = []
all_summaries = []
def benchmark(experiment, factories):
    names = list(factories)
    for name in names:
        assert canonical(factories[name]().collect()) == reference
    records = []
    for repeat in range(REPEATS):
        offset = repeat % len(names)
        for name in names[offset:] + names[:offset]:
            query = factories[name]()
            prefix = f'{experiment}_{name}_{repeat+1}'
            (RUN_DIR / f'{prefix}_before.txt').write_text(capture_plan(query), encoding='utf-8')
            started = perf_counter()
            rows = query.collect()
            seconds = perf_counter() - started
            assert canonical(rows) == reference, f'Hasil berubah: {prefix}'
            (RUN_DIR / f'{prefix}_after.txt').write_text(capture_plan(query), encoding='utf-8')
            records.append({'experiment': experiment, 'variant': name,
                            'repeat': repeat+1, 'seconds': seconds, 'correct': True})
    summary = []
    for name in names:
        values = [r['seconds'] for r in records if r['variant'] == name]
        summary.append({'experiment': experiment, 'variant': name,
                        'median_seconds': statistics.median(values),
                        'min_seconds': min(values), 'max_seconds': max(values),
                        'runs': len(values), 'correct': True})
    all_trials.extend(records)
    all_summaries.extend(summary)
    save_json(f'{experiment}_trials.json', records)
    save_json(f'{experiment}_summary.json', summary)
    pprint(summary)
    return summary


## 8. Hands-on: sort-merge dan broadcast join
Broadcast cocok untuk sisi join yang kecil setelah baris dan kolom yang diperlukan dipilih.
Ukuran file di disk belum tentu sama dengan ukuran tabel di memori.

Kedua varian di bawah memakai data dan agregasi yang sama. Automatic broadcast dan AQE
dimatikan agar hint broadcast menjadi perubahan yang sedang diuji.


In [7]:
def join_query(broadcast=False):
    configure()
    dimension = F.broadcast(customers) if broadcast else customers
    return aggregate(orders.join(dimension, 'customer_id', 'inner'))

join_summary = benchmark('join', {'sort_merge': lambda: join_query(False),
                                  'broadcast': lambda: join_query(True)})
normal_plan = capture_plan(join_query(False))
broadcast_plan = capture_plan(join_query(True))
assert 'SortMergeJoin' in normal_plan
assert 'BroadcastHashJoin' in broadcast_plan
print('Plan baseline: SortMergeJoin')
print('Plan kandidat: BroadcastHashJoin')

completed_checks.append('sort-merge vs broadcast')


[{'correct': True,
  'experiment': 'join',
  'max_seconds': 0.36058309999953053,
  'median_seconds': 0.3194893999998385,
  'min_seconds': 0.30908180000005814,
  'runs': 3,
  'variant': 'sort_merge'},
 {'correct': True,
  'experiment': 'join',
  'max_seconds': 0.173229199999696,
  'median_seconds': 0.1619741000004069,
  'min_seconds': 0.15579689999958646,
  'runs': 3,
  'variant': 'broadcast'}]
Plan baseline: SortMergeJoin
Plan kandidat: BroadcastHashJoin


**Cara membaca hasil:** kedua varian harus menghasilkan `correct=True`.
Periksa median dan rentangnya. Broadcast dapat mengurangi shuffle pada join, tetapi
agregasi setelah join masih dapat membutuhkan shuffle. Data kecil dapat membuat waktu
lebih banyak dipengaruhi overhead dibanding volume data.


## 9. Hands-on: repartition dan coalesce
Repartition membagi ulang data dan dapat menambah atau mengurangi jumlah partisi.
Coalesce digunakan untuk mengurangi partisi. Pengurangan yang terlalu besar dapat
membuat lebih sedikit task mengerjakan seluruh data.

Contoh ini tidak menulis file. Jumlah baris dan hasil agregasi diperiksa setelah perubahan.


In [8]:
configure()
repartitioned = direct.repartition(8, 'customer_id')
coalesced = repartitioned.coalesce(2)
print('Partisi awal:', direct.rdd.getNumPartitions())
print('Sesudah repartition:', repartitioned.rdd.getNumPartitions())
print('Sesudah coalesce:', coalesced.rdd.getNumPartitions())
assert repartitioned.rdd.getNumPartitions() == 8
assert coalesced.rdd.getNumPartitions() == 2
assert canonical(aggregate(repartitioned).collect()) == reference
assert canonical(aggregate(coalesced).collect()) == reference
print('Hasil agregasi tetap sama.')

completed_checks.append('repartition/coalesce')


Partisi awal: 4
Sesudah repartition: 8
Sesudah coalesce: 2


Hasil agregasi tetap sama.


## 10. Hands-on: jumlah partisi shuffle
`spark.sql.shuffle.partitions` mengatur jumlah partisi shuffle awal untuk operasi SQL yang
terkait. Nilai ini berbeda dari jumlah folder partisi pada dataset.

Kita membandingkan 2, 8, dan 32 partisi dengan AQE mati. Angka ini dipilih untuk latihan.
Jumlah partisi yang sesuai pada produksi bergantung pada data dan resource.


In [9]:
def partition_query(n):
    configure(partitions=n)
    return aggregate(orders.join(customers, 'customer_id', 'inner'))

partition_summary = benchmark('partitions', {
    'p2': lambda: partition_query(2),
    'p8': lambda: partition_query(8),
    'p32': lambda: partition_query(32),
})

completed_checks.append('shuffle partitions')


[{'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.1253801000002568,
  'median_seconds': 0.11056540000026871,
  'min_seconds': 0.10183239999969373,
  'runs': 3,
  'variant': 'p2'},
 {'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.22284589999981108,
  'median_seconds': 0.22224329999971815,
  'min_seconds': 0.21190529999967112,
  'runs': 3,
  'variant': 'p8'},
 {'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.691242500000044,
  'median_seconds': 0.6906105999996726,
  'min_seconds': 0.66891779999969,
  'runs': 3,
  'variant': 'p32'}]


**Cara membaca hasil:** partisi yang banyak menghasilkan lebih banyak pekerjaan penjadwalan.
Partisi yang terlalu sedikit dapat mengurangi paralelisme. Pada data kecil, 32 partisi
bisa lebih lambat daripada 2 partisi. Gunakan hasil pengukuran yang muncul di perangkat Anda.


## 11. Hands-on: Adaptive Query Execution
AQE memakai statistik yang tersedia saat query berjalan. Salah satu kemampuannya adalah
menggabungkan partisi shuffle yang kecil. Untuk percobaan ini, kedua varian mulai dari
64 partisi dan adaptive broadcast dimatikan agar perubahan partisi lebih mudah dibaca.

Plan sebelum action dapat berbeda dari final plan. Karena itu, kita membaca keduanya.


In [10]:
def aqe_query(enabled):
    configure(partitions=64, aqe=enabled)
    return aggregate(orders.join(customers, 'customer_id', 'inner'))

aqe_summary = benchmark('aqe', {'off': lambda: aqe_query(False), 'on': lambda: aqe_query(True)})
aqe_plan = (RUN_DIR / 'aqe_on_1_after.txt').read_text(encoding='utf-8')
assert 'isFinalPlan=true' in aqe_plan
print('Baris plan yang menunjukkan penyesuaian:')
for line in aqe_plan.splitlines():
    if any(term in line for term in ['AdaptiveSparkPlan', 'AQEShuffleRead', 'coalesced', 'isFinalPlan']):
        print(line)

completed_checks.append('AQE final plan')


[{'correct': True,
  'experiment': 'aqe',
  'max_seconds': 1.688088500000049,
  'median_seconds': 1.3405490999994072,
  'min_seconds': 1.2887230999995154,
  'runs': 3,
  'variant': 'off'},
 {'correct': True,
  'experiment': 'aqe',
  'max_seconds': 0.3662843000001885,
  'median_seconds': 0.35584879999987606,
  'min_seconds': 0.3538936000004469,
  'runs': 3,
  'variant': 'on'}]
Baris plan yang menunjukkan penyesuaian:
AdaptiveSparkPlan (31)
      +- AQEShuffleRead (19)
                        :  +- AQEShuffleRead (6)
                           +- AQEShuffleRead (12)
(6) AQEShuffleRead
Arguments: coalesced
(12) AQEShuffleRead
Arguments: coalesced
(19) AQEShuffleRead
Arguments: coalesced
(31) AdaptiveSparkPlan
Arguments: isFinalPlan=true


## 12. Hands-on: cache dan biaya materialisasi
Cache berguna ketika hasil perantara digunakan kembali. Kita menambahkan transformasi hash
dan menjalankan tiga query pada hasil tersebut. Varian cache menghitung biaya pengisian
cache ditambah tiga query. Varian tanpa cache menjalankan tiga query langsung.

Timer bagian ini mencakup keseluruhan pekerjaan tersebut, termasuk penyusunan ketiga query.
Hasil setiap query dibandingkan dengan referensi Python. Cache dilepas setelah tiap trial.


In [11]:
def cache_trial(use_cache):
    configure()
    fingerprint = F.col('order_id').cast('string')
    for _ in range(4):
        fingerprint = F.sha2(fingerprint, 256)
    base = direct.withColumn('fingerprint', fingerprint)
    if use_cache:
        base = base.cache()
    try:
        started = perf_counter()
        materialize = 0.0
        if use_cache:
            base.count()
            materialize = perf_counter() - started
        results = []
        for k in range(3):
            query = (base.filter(F.col('order_id') % 3 == k).groupBy('region')
                     .agg(F.sum('amount').alias('revenue'),
                          F.sum(F.length('fingerprint')).alias('fingerprint_chars'))
                     .select('region', 'revenue', 'fingerprint_chars'))
            results.append(query.collect())
        seconds = perf_counter() - started
        assert [canonical(rows) for rows in results] == cache_reference
        return seconds, materialize
    finally:
        if use_cache:
            base.unpersist(blocking=True)

cache_trial(False)
cache_trial(True)
cache_records = []
for repeat in range(REPEATS):
    for use_cache in ([False, True] if repeat % 2 == 0 else [True, False]):
        seconds, materialize = cache_trial(use_cache)
        cache_records.append({'experiment': 'cache', 'variant': 'cache' if use_cache else 'no_cache',
                              'repeat': repeat+1, 'seconds': seconds,
                              'materialize_seconds': materialize, 'correct': True})
cache_summary = []
for name in ['no_cache', 'cache']:
    values = [r['seconds'] for r in cache_records if r['variant'] == name]
    cache_summary.append({'experiment': 'cache', 'variant': name,
                          'median_seconds': statistics.median(values),
                          'min_seconds': min(values), 'max_seconds': max(values),
                          'runs': len(values), 'correct': True})
all_trials.extend(cache_records)
all_summaries.extend(cache_summary)
save_json('cache_trials.json', cache_records)
save_json('cache_summary.json', cache_summary)
pprint(cache_summary)

completed_checks.append('cache results')


[{'correct': True,
  'experiment': 'cache',
  'max_seconds': 0.388919299999543,
  'median_seconds': 0.38340839999909804,
  'min_seconds': 0.34117979999973613,
  'runs': 3,
  'variant': 'no_cache'},
 {'correct': True,
  'experiment': 'cache',
  'max_seconds': 0.4205000000001746,
  'median_seconds': 0.3869203000003836,
  'min_seconds': 0.321879799999806,
  'runs': 3,
  'variant': 'cache'}]


**Cara membaca hasil:** bandingkan waktu total kedua varian. Jika cache lebih lambat,
biaya pengisian cache belum terbayar oleh tiga query pada data tersebut. Tab Storage
dapat kosong di akhir karena notebook sudah menjalankan `unpersist()`.


## 13. Hands-on: distribusi skew
Kita memberi satu key yang sama kepada 80% baris, lalu melakukan repartition berdasarkan
key tersebut. Hash partitioning mengirim key yang sama ke partisi yang sama.
Jumlah partisi yang lebih banyak tidak otomatis membagi satu key dominan.


In [12]:
configure()
skewed = orders.withColumn('customer_id',
    F.when(F.col('order_id') % 10 < 8, F.lit(0)).otherwise(F.col('customer_id')))
distribution = (skewed.repartition(8, 'customer_id')
                .select(F.spark_partition_id().alias('partition_id'))
                .groupBy('partition_id').count())
distribution_rows = sorted([r.asDict() for r in distribution.collect()], key=lambda r: r['partition_id'])
assert sum(r['count'] for r in distribution_rows) == N_ROWS
assert max(r['count'] for r in distribution_rows) >= N_ROWS * 0.8
save_json('skew_distribution.json', distribution_rows)
pprint(distribution_rows)
print('Total baris:', sum(r['count'] for r in distribution_rows))

completed_checks.append('skew total rows')


[{'count': 210, 'partition_id': 0},
 {'count': 290, 'partition_id': 1},
 {'count': 240, 'partition_id': 2},
 {'count': 280, 'partition_id': 3},
 {'count': 240, 'partition_id': 4},
 {'count': 8250, 'partition_id': 5},
 {'count': 190, 'partition_id': 6},
 {'count': 300, 'partition_id': 7}]
Total baris: 10000


## 14. Hands-on: filter dan select
Filter memilih baris dan select memilih kolom. Optimizer dapat memindahkan atau
menyederhanakan operasi tersebut. Kita memeriksa hasil dan melihat plan-nya.

Data notebook berasal dari `spark.range`. Contoh ini memperlihatkan Filter dan Project,
sedangkan pembuktian partition pruning dan predicate pushdown pada file memerlukan
sumber seperti Parquet. Notebook ini tidak membaca atau menulis dataset Parquet.


In [13]:
filtered = orders.filter(F.col('amount') >= 5000).select('order_id', 'amount')
expected_count = sum(1 for i in range(N_ROWS) if (i % 97 + 1) * 100 >= 5000)
assert filtered.count() == expected_count
assert filtered.columns == ['order_id', 'amount']
filtered.explain('formatted')
filtered.show(5, truncate=False)
print('Jumlah baris sesuai referensi:', expected_count)

completed_checks.append('filter/select')


== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 10000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((((id#0L % 97) + 1) * 100) >= 5000)

(3) Project [codegen id : 1]
Output [2]: [id#0L AS order_id#1L, (((id#0L % 97) + 1) * 100) AS amount#3L]
Input [1]: [id#0L]


+--------+------+
|order_id|amount|
+--------+------+
|49      |5000  |
|50      |5100  |
|51      |5200  |
|52      |5300  |
|53      |5400  |
+--------+------+
only showing top 5 rows
Jumlah baris sesuai referensi: 4944


## 15. Hands-on: pemeriksaan hasil yang salah
Kandidat berikut sengaja menghilangkan region 0. Assertion harus menangkap perbedaan.
Error yang diharapkan ditangani di dalam sel, sehingga Run All dapat berlanjut.


In [14]:
wrong = canonical(aggregate(direct.filter(F.col('region') != 0)).collect())
failure_caught = False
try:
    assert wrong == reference, 'Hasil kandidat kehilangan region 0.'
except AssertionError as error:
    failure_caught = True
    print('Pemeriksaan menangkap kesalahan:', error)
assert failure_caught, 'Validasi gagal mendeteksi perubahan hasil.'
save_json('failure.json', {'caught': failure_caught, 'expected': reference, 'actual': wrong})

completed_checks.append('intentional failure')


Pemeriksaan menangkap kesalahan: Hasil kandidat kehilangan region 0.


## 16. Menyimpan ringkasan pengukuran
Speedup dihitung dari waktu baseline dibagi waktu kandidat. Penurunan durasi dihitung
dari selisih waktu dibagi baseline. Nilainya bisa negatif jika kandidat lebih lambat.
File hasil berisi angka dari run ini, bukan angka ilustrasi pada PPT.


In [15]:
baseline_median = next(r['median_seconds'] for r in join_summary if r['variant'] == 'sort_merge')
broadcast_median = next(r['median_seconds'] for r in join_summary if r['variant'] == 'broadcast')
speedup = baseline_median / broadcast_median
reduction = (baseline_median - broadcast_median) / baseline_median * 100
print(f'Join speedup: {speedup:.3f} kali')
print(f'Perubahan durasi: {reduction:.2f}% lebih singkat (negatif berarti lebih lama)')
assert all(row['correct'] for row in all_trials)
save_json('all_trials.json', all_trials)
save_json('all_summaries.json', all_summaries)
with (RUN_DIR / 'summary.csv').open('w', newline='', encoding='utf-8') as handle:
    fields = ['experiment', 'variant', 'median_seconds', 'min_seconds', 'max_seconds', 'runs', 'correct']
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(all_summaries)
pprint(all_summaries)
print('File hasil tersimpan di:', RUN_DIR)


Join speedup: 1.972 kali
Perubahan durasi: 49.30% lebih singkat (negatif berarti lebih lama)
[{'correct': True,
  'experiment': 'join',
  'max_seconds': 0.36058309999953053,
  'median_seconds': 0.3194893999998385,
  'min_seconds': 0.30908180000005814,
  'runs': 3,
  'variant': 'sort_merge'},
 {'correct': True,
  'experiment': 'join',
  'max_seconds': 0.173229199999696,
  'median_seconds': 0.1619741000004069,
  'min_seconds': 0.15579689999958646,
  'runs': 3,
  'variant': 'broadcast'},
 {'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.1253801000002568,
  'median_seconds': 0.11056540000026871,
  'min_seconds': 0.10183239999969373,
  'runs': 3,
  'variant': 'p2'},
 {'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.22284589999981108,
  'median_seconds': 0.22224329999971815,
  'min_seconds': 0.21190529999967112,
  'runs': 3,
  'variant': 'p8'},
 {'correct': True,
  'experiment': 'partitions',
  'max_seconds': 0.691242500000044,
  'median_seconds': 0.690610

## 17. Metrik CPU, memori, dan disk

Durasi saja belum menjelaskan penyebab query lambat. Kita membaca metrik dari
REST API Spark UI, yaitu alamat lokal yang menyediakan data stage dan task dalam JSON.
Tidak diperlukan layanan luar atau akun cloud.

| Metrik | Arti | Batas pembacaan |
|---|---|---|
| `seconds` | Waktu yang berlalu pada komputer selama operasi | Berbeda dari jumlah waktu semua task |
| `executor_cpu_seconds` | Jumlah waktu CPU thread task | Bukan persentase CPU komputer; tidak mencakup seluruh pekerjaan driver |
| `peak_task_execution_bytes` | Puncak memori operasi tertinggi di antara task | Bukan total RAM proses atau jumlah RAM seluruh executor |
| `memory_spill_bytes` | Ukuran representasi data di memori yang di-spill | Bisa lebih besar daripada file disk akibat perbedaan representasi/kompresi |
| `disk_spill_bytes` | Byte data perantara yang ditulis karena spill | Berbeda dari ukuran output Parquet |
| `input_bytes` | Byte masukan yang tercatat oleh task | Bukan seluruh lalu lintas storage/network |

Modul `spark_storage_lab.py` berada di samping notebook. Fungsi `measure()` memberi
nama pada kelompok job, mengukur action, lalu mengambil stage dan task yang sesuai.
Pengambilan metrik serta assertion dilakukan di luar timer. Data task mentah disimpan
agar angka ringkasan dapat diperiksa. Metrik yang hilang menyebabkan error, bukan diisi nol.

Setiap praktik memakai folder baru. Menjalankan ulang sel praktik tanpa Run All
akan ditolak jika folder tujuannya sudah ada. Ini mencegah hasil sebelumnya tertimpa.


In [16]:
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
from spark_storage_lab import run_spill, run_small_files, write_report
STORAGE_DIR = RUN_DIR / 'storage'
STORAGE_DIR.mkdir(exist_ok=False)
print('Praktik tambahan:', STORAGE_DIR)
print('Metrik berasal dari aplikasi:', spark.sparkContext.applicationId)


Praktik tambahan: F:\programming\ngajar\full_stack_bangalore\sesi 17 Data Lake MinIO\sesi16\lab\notebook_outputs\20261025_164931_d6c23457\storage
Metrik berasal dari aplikasi: local-1792921781611


## 18. Spill: data perantara ditulis ke disk

Spill terjadi ketika sebuah operasi perlu memindahkan sebagian data perantara dari
memori ke disk. Ini membantu pekerjaan tetap berjalan, tetapi menambah baca/tulis disk.
Periksa spill saat sort, agregasi, join, atau window lambat; hubungkan dengan ukuran
partisi, skew, memori operasi, dan durasi task. Spill tidak selalu berarti job gagal.

Buffer adalah ruang sementara untuk menampung baris selama operasi.

**Window** menghitung nilai untuk sekumpulan baris sambil mempertahankan baris asal.
Contoh kita menghitung jumlah seluruh `id`, lalu menampilkan total itu pada setiap baris.
`Window.orderBy('id')` mengurutkan baris; `rowsBetween` menentukan cakupan perhitungan.
`unboundedPreceding` sampai `unboundedFollowing` mencakup semua baris.

Demo dibatasi maksimal 10.000 baris karena hasil dikumpulkan ke driver untuk diperiksa
satu per satu. Dua varian memakai query, data, satu kelompok window, serta ambang
in-memory 128 baris yang sama. Hanya ambang spill diubah:

- `normal_buffer`: 2.147.483.647 baris.
- `forced_spill`: 512 baris agar penulisan spill terlihat pada data kecil.

Nama konfigurasi: `spark.sql.windowExec.buffer.spill.threshold`.
Ini **konfigurasi internal khusus demo Spark 4.0.1**, bukan ukuran RAM dan bukan anjuran
produksi. Demo tidak sengaja menghabiskan RAM. Pengaturan awal dipulihkan dalam `finally`.
Setiap varian dipanaskan sekali, lalu diukur tiga kali dengan urutan bergantian.

Yang diperiksa: jumlah baris tetap sama, setiap `id` benar, setiap total sama dengan
`n*(n-1)//2`, dan disk spill varian paksa harus lebih dari nol. Di UI, buka **Stages**,
pilih stage dari JSON, lalu baca **Spill (Memory)** dan **Spill (Disk)** pada detail task.

Untuk produksi: kurangi kolom/baris lebih awal jika hasil tetap sesuai; periksa skew;
uji partisi yang lebih seimbang. Menambah RAM baru dipertimbangkan setelah penyebabnya
jelas. Menaikkan ambang demo tidak otomatis menyelesaikan tekanan memori produksi.


In [17]:
spill_result = run_spill(spark, STORAGE_DIR, rows=N_ROWS, repeats=REPEATS)
pprint(spill_result['summary'])
assert all(r['correct'] for r in spill_result['trials'])
assert all(r['disk_spill_bytes'] > 0 for r in spill_result['trials']
           if r['variant'] == 'forced_spill')
completed_checks.append('spill observed and results equal')


[{'correct': True,
  'max_seconds': 0.09392200000002049,
  'median_disk_spill_bytes': 0,
  'median_executor_cpu_seconds': 0.046875,
  'median_input_bytes': 0,
  'median_memory_spill_bytes': 0,
  'median_peak_task_execution_bytes': 8912880,
  'median_seconds': 0.0724051999995936,
  'median_shuffle_read_bytes': 51717,
  'median_shuffle_write_bytes': 51717,
  'median_task_elapsed_seconds': 0.025,
  'median_tasks': 3,
  'min_seconds': 0.0703350000003411,
  'runs': 3,
  'variant': 'normal_buffer'},
 {'correct': True,
  'max_seconds': 0.2139286000001448,
  'median_disk_spill_bytes': 52374,
  'median_executor_cpu_seconds': 0.09375,
  'median_input_bytes': 0,
  'median_memory_spill_bytes': 159383248,
  'median_peak_task_execution_bytes': 8912880,
  'median_seconds': 0.18481860000065353,
  'median_shuffle_read_bytes': 51717,
  'median_shuffle_write_bytes': 51717,
  'median_task_elapsed_seconds': 0.121,
  'median_tasks': 3,
  'min_seconds': 0.13644560000011552,
  'runs': 3,
  'variant': 'forced_

## 19. Small files: banyak file dengan isi sedikit

Parquet adalah format kolom yang dipakai menyimpan data analitik. Setiap file memiliki
metadata, yaitu informasi struktur file seperti schema dan jumlah baris; membuka banyak file memerlukan pekerjaan tambahan meskipun total data kecil.
File dan partisi Spark berbeda: satu task baca dapat membaca beberapa file kecil.
Jumlah file kecil yang bermasalah bergantung pada storage, ukuran data, dan pola query.

Praktik ini membuat 32 file Parquet dari data transaksi, lalu menggabungkannya menjadi
2 file pada lokasi baru. Pemadatan ini disebut **compaction**. Kode intinya:

```python
source.repartition(32).write.mode('errorifexists').parquet(uri_many)
spark.read.parquet(uri_many).coalesce(2).write.mode('errorifexists').parquet(uri_compact)
```

`repartition(32)` membagi ulang data dengan shuffle. `coalesce(2)` mengurangi partisi
sebelum penulisan. Pada data lab tanpa pemisahan folder/rotasi file, hasilnya 32 dan 2
file data. `errorifexists` menolak menimpa lokasi yang sudah ada. `_SUCCESS` dan file
checksum tidak dihitung sebagai file data Parquet.

Kedua lokasi dibaca dengan agregasi yang sama: jumlah transaksi, jumlah order_id,
dan total amount. Tiga hasil itu diperiksa terhadap rumus Python. `exceptAll` dua arah
juga memeriksa bahwa isi lengkap dan jumlah duplikat tetap sama setelah pemadatan.

Bandingkan jumlah file, ukuran minimum/rata-rata/maksimum, durasi, CPU task, dan metrik
disk. Waktu pembentukan 32 file dan pemadatan diukur terpisah dari benchmark baca.
Timer baca mencakup pembuatan reader, listing/schema, perencanaan, dan action agregasi.
Cache OS tidak dikosongkan. Dua file bukan target ukuran untuk produksi: file terlalu
sedikit dapat membatasi pekerjaan paralel. Penulisan Parquet di Windows memerlukan
dukungan Hadoop lokal yang sesuai; bila gagal pada winutils, gunakan lingkungan Linux
atau Docker yang disiapkan pengajar. Jangan mengganti hasil gagal dengan angka contoh.


In [18]:
files_result = run_small_files(spark, STORAGE_DIR, rows=N_ROWS, repeats=REPEATS)
pprint(files_result['inventory'])
pprint(files_result['summary'])
assert files_result['exact_content_equal']
assert all(r['correct'] for r in files_result['trials'])
completed_checks.append('small files exact content equal')


{'compact': {'average_bytes': 25880.0,
             'files': 2,
             'max_bytes': 25913,
             'min_bytes': 25847,
             'total_bytes': 51760},
 'many': {'average_bytes': 2825.625,
          'files': 32,
          'max_bytes': 2850,
          'min_bytes': 2809,
          'total_bytes': 90420}}
[{'correct': True,
  'max_seconds': 0.2117428999999902,
  'median_disk_spill_bytes': 0,
  'median_executor_cpu_seconds': 0.109375,
  'median_input_bytes': 126108,
  'median_memory_spill_bytes': 0,
  'median_peak_task_execution_bytes': 0,
  'median_seconds': 0.20055020000017976,
  'median_shuffle_read_bytes': 142,
  'median_shuffle_write_bytes': 142,
  'median_task_elapsed_seconds': 0.123,
  'median_tasks': 4,
  'min_seconds': 0.1903149999998277,
  'runs': 3,
  'variant': 'many'},
 {'correct': True,
  'max_seconds': 0.15262780000011844,
  'median_disk_spill_bytes': 0,
  'median_executor_cpu_seconds': 0.015625,
  'median_input_bytes': 54768,
  'median_memory_spill_bytes': 0,
 

## 20. Membandingkan resource dan membuat Spark Optimization Report

Bandingkan waktu baca dengan CPU, memori operasi, jumlah file, dan biaya pemadatan.
Jika baca kandidat lebih cepat, hitung perkiraan jumlah pembacaan untuk menutup waktu
pemadatan: `ceil(waktu_pemadatan / (waktu_baseline - waktu_kandidat))`.
Jika selisihnya nol atau negatif, penghematan tersebut belum ada pada run ini.

Untuk memperkenalkan biaya uang, gunakan **tarif ilustrasi** Rp5.000/jam pada mesin yang
sama. Estimasi `durasi_detik / 3600 * tarif_per_jam`. Ini bukan tarif penyedia cloud.
Startup, waktu idle, storage, jaringan, dan pembulatan tagihan belum dihitung. Mesin
yang terus menyala belum tentu menjadi lebih murah hanya karena query selesai cepat.

Laporan otomatis berisi masalah, lingkungan, hipotesis, variabel kontrol, hasil,
pemeriksaan kebenaran, biaya pemadatan, keputusan, keterbatasan, eksperimen berikutnya,
dan penerapan pada capstone. Keputusan mengikuti angka run ini, termasuk bila kandidat
lebih lambat. Semua hasil mentah tetap tersedia di folder run.

Pada capstone transaksi ke lake, uji kandidat di lokasi baru. Periksa schema, jumlah
transaksi, duplikasi, dan total penjualan per tanggal. Jika hasil berubah atau durasi
melewati SLA (batas waktu layanan), gunakan lagi lokasi sumber. Langkah penerapan ini
dibahas sebagai rancangan; notebook tidak mengakses MinIO atau mengganti pipeline asli.


In [19]:
HOURLY_RATE_ILLUSTRATION = 5000.0
resource_comparison = write_report(spark, STORAGE_DIR, spill_result, files_result,
                                   hourly_rate=HOURLY_RATE_ILLUSTRATION)
pprint(resource_comparison)
report_path = STORAGE_DIR / 'Spark_Optimization_Report.md'
assert report_path.is_file()
completed_checks.append('resource comparison and report')
print('Laporan lengkap:', report_path)
print(report_path.read_text(encoding='utf-8'))


{'baseline': {'correct': True,
              'max_seconds': 0.2117428999999902,
              'median_disk_spill_bytes': 0,
              'median_executor_cpu_seconds': 0.109375,
              'median_input_bytes': 126108,
              'median_memory_spill_bytes': 0,
              'median_peak_task_execution_bytes': 0,
              'median_seconds': 0.20055020000017976,
              'median_shuffle_read_bytes': 142,
              'median_shuffle_write_bytes': 142,
              'median_task_elapsed_seconds': 0.123,
              'median_tasks': 4,
              'min_seconds': 0.1903149999998277,
              'runs': 3,
              'variant': 'many'},
 'baseline_cost_illustration': 0.2785419444446941,
 'candidate': {'correct': True,
               'max_seconds': 0.15262780000011844,
               'median_disk_spill_bytes': 0,
               'median_executor_cpu_seconds': 0.015625,
               'median_input_bytes': 54768,
               'median_memory_spill_bytes': 0,
         

## 21. Menutup sesi
Sel ini menghentikan Spark yang dibuat oleh notebook dan mengembalikan environment
kernel ke kondisi awal. Spark UI tidak lagi tersedia setelah sesi berhenti.
Jika proses sebelumnya terputus karena error, jalankan sel ini untuk menutup sesi,
atau restart kernel sebelum mencoba ulang.


In [20]:
try:
    spark.stop()
finally:
    for key, old_value in env_before.items():
        if old_value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = old_value
completed = set(completed_checks) == set(['reference', 'sort-merge vs broadcast', 'repartition/coalesce', 'shuffle partitions', 'AQE final plan', 'cache results', 'skew total rows', 'filter/select', 'intentional failure', 'spill observed and results equal', 'small files exact content equal', 'resource comparison and report']) and len(all_trials) == 9 * REPEATS and len(spill_result['trials']) == 2 * REPEATS and len(files_result['trials']) == 2 * REPEATS
save_json('verification.json', {'completed': completed, 'rows': N_ROWS, 'repeats': REPEATS,
    'checks': completed_checks,
    'measured_trials': len(all_trials) + (len(spill_result['trials']) + len(files_result['trials']) if 'spill_result' in globals() and 'files_result' in globals() else 0), 'spark_stopped': True})
print('Spark dihentikan dan environment dipulihkan.')
print('Seluruh tahap lengkap:', completed)


Spark dihentikan dan environment dipulihkan.
Seluruh tahap lengkap: True


## Jika ada kendala

| Gejala | Langkah berikutnya |
|---|---|
| `No module named pyspark` | Pilih kernel `.venv` lab dan pasang `requirements-notebook.txt` di environment itu |
| Java tidak ditemukan atau versinya lama | Periksa `java -version`, pilih Java 17 pada PATH terminal lab, lalu restart kernel |
| SparkContext sudah aktif | Gunakan Restart Kernel lalu Run All |
| UI tidak terbuka | Buka URL yang dicetak sel Spark sebelum sel penutup berjalan |
| Run terasa lama | Mulai dari 10.000 baris. Tutup aplikasi berat yang tidak diperlukan |
| Waktu berbeda dari contoh | Periksa data, jumlah pengukuran, dan beban perangkat. Hasil antar-laptop memang bisa berbeda |
| Kernel berhenti di tengah | Restart kernel lalu jalankan dari awal. Folder run lama tetap tersedia |

## Referensi
- [Komponen Spark](https://spark.apache.org/docs/4.0.1/cluster-overview.html)
- [Performance tuning dan AQE](https://spark.apache.org/docs/4.0.1/sql-performance-tuning.html)
- [Spark UI](https://spark.apache.org/docs/4.0.1/web-ui.html)
- [repartition](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)
- [coalesce](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html)
- [cache](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html)

Notebook ini menguji data sintetis lokal, termasuk spill yang sengaja dipicu. Kehabisan
memori, cluster beberapa mesin, dan akses MinIO tidak dijalankan dalam latihan.
